# Übung 03 – Musikgenerierung mit RNNs 🎵

**Einführung in Deep Learning (SoSe26) – Abgabe 3**

In diesem Notebook trainieren wir ein **Rekurrentes Neuronales Netz (LSTM)**, das irische Volksmusik in **ABC-Notation** lernt und anschließend **neue Melodien generiert**.

**Idee:** Musik in ABC-Notation ist einfach Text. Wir zerlegen den Text in Tokens (hier: einzelne Zeichen), und das Netz lernt, **das jeweils nächste Zeichen vorherzusagen**. Wenn das Netz das gut kann, lassen wir es Zeichen für Zeichen "weiterschreiben" → neue Musik!

**Datensatz:** [IrishMAN](https://huggingface.co/datasets/sander-wood/irishman) – irische Volkslieder in ABC-Notation.

**Inhalt:**
1. Setup & Hyperparameter (+ wandb)
2. Task 1: Daten laden & Tokenisierung
3. Task 2: Modell (Embedding → LSTM → Linear)
4. Task 3: Training
5. Task 5: Evaluation (Top-1 / Top-5 Accuracy)
6. Task 4: Musikgenerierung
7. Bonus 1: Ablation LSTM vs. GRU
8. Bonus 2: Validity Check (ABC-Syntax)
9. Bonus 3: Audio-Ausgabe (.wav)
10. Trainingskurven & Export für die Präsentation
11. Fazit

## 1. Setup

Wir benutzen PyTorch, den `datasets`-Loader von Hugging Face (für IrishMAN) und **Weights & Biases (wandb)** zum Visualisieren des Trainings.

In [1]:
%pip install --quiet datasets wandb python-pptx

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Imports – alles Standard-Werkzeuge aus der Vorlesung
import json
import random
import wave
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from datasets import load_dataset
import matplotlib.pyplot as plt
import wandb

# Auf der GPU trainieren, falls verfügbar (sonst CPU)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Verwendetes Gerät:", DEVICE)

# Zufalls-Seeds, damit die Ergebnisse reproduzierbar sind
random.seed(42)
torch.manual_seed(42)

Verwendetes Gerät: cpu


## 2. Hyperparameter & wandb-Konfiguration

Jeder Trainingslauf wird als **Run** im wandb-Projekt gespeichert (Entity = Team des Kurses, Project = diese Abgabe). So können wir Hyperparameter und Kurven aller Versuche auf [wandb.ai](https://wandb.ai) vergleichen.

In [3]:
# === wandb-Einstellungen (siehe WANDB_SETUP_GUIDE.md) ===
ENTITY = "eidl-thm"                                   # Team/Organisation des Kurses
PROJECT = "4.block_Jordan_Pokem_Leslie_Tsafack_RNN"   # NEUES Projekt für diese Abgabe

# === Hyperparameter ===
NUM_TUNES     = 8000    # wie viele Musikstücke wir aus dem Datensatz verwenden
SEQ_LEN       = 100     # Länge einer Trainingssequenz (in Zeichen)
SCHRITT       = 50      # Schrittweite des gleitenden Fensters
BATCH_SIZE    = 128
EMBED_DIM     = 64      # Größe der Embedding-Vektoren
HIDDEN_SIZE   = 256     # Größe des Hidden States im RNN
NUM_LAYERS    = 2       # Anzahl gestapelter RNN-Schichten
LEARNING_RATE = 1e-3
NUM_EPOCHS    = 10
GEN_LAENGE    = 500     # wie viele Zeichen wir später generieren

# Alle Hyperparameter in einem Dictionary – wird an wandb übergeben,
# damit wir jeden Run später genau nachvollziehen können.
config = {
    "model": "LSTM",
    "num_tunes": NUM_TUNES,
    "seq_len": SEQ_LEN,
    "schritt": SCHRITT,
    "batch_size": BATCH_SIZE,
    "embed_dim": EMBED_DIM,
    "hidden_size": HIDDEN_SIZE,
    "num_layers": NUM_LAYERS,
    "learning_rate": LEARNING_RATE,
    "num_epochs": NUM_EPOCHS,
}

# Einmalige Anmeldung bei wandb (API-Key wird lokal gespeichert)
wandb.login()
print(f"wandb: {ENTITY}/{PROJECT}")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\pokem\_netrc.
wandb: Currently logged in as: pokemtezo5 (eidl-thm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: eidl-thm/4.block_Jordan_Pokem_Leslie_Tsafack_RNN


wandb: eidl-thm/4.block_Jordan_Pokem_Leslie_Tsafack_RNN


## 3. Task 1 – Daten laden

**ABC-Notation** ist ein Textformat für Musik: ein paar Kopfzeilen (`M:` Taktart, `L:` Grundnotenlänge, `K:` Tonart), danach die Melodie – Buchstaben `A–G`/`a–g` sind Noten, `|` sind Taktstriche, `z` ist eine Pause.

In [4]:
# IrishMAN-Datensatz von Hugging Face laden (wird beim ersten Mal heruntergeladen)
from datasets import load_dataset

ds = load_dataset("sander-wood/irishman")
print(ds)

# Wir nehmen ein Teil-Set aus dem Trainings-Split, damit das Training schnell bleibt
stuecke = ds["train"].select(range(NUM_TUNES))["abc notation"]

print("\n=== Beispiel-Stück aus dem Datensatz ===")
print(stuecke[0])

DatasetDict({
    train: Dataset({
        features: ['abc notation', 'control code'],
        num_rows: 214122
    })
    validation: Dataset({
        features: ['abc notation', 'control code'],
        num_rows: 2162
    })
})

=== Beispiel-Stück aus dem Datensatz ===
X:1
L:1/8
M:4/4
K:Emin
|: E2 EF E2 EF | DEFG AFDF | E2 EF E2 B2 |1 efe^d e2 e2 :|2 efe^d e3 B |: e2 ef g2 fe | 
 defg afdf |1 e2 ef g2 fe | efe^d e3 B :|2 g2 bg f2 af | efe^d e2 e2 ||


### Tokenisierung (Zeichen-Ebene)

Wie in Vorlesung 9: Text muss für ein neuronales Netz in **Zahlen-IDs** umgewandelt werden.

- **Encoding:** Text → IDs   |   **Decoding:** IDs → Text
- Wir benutzen einen **Zeichen-Tokenizer** (jedes Zeichen = 1 Token). Vorteil: sehr einfach und **kein OOV-Problem** (es kann keine unbekannten Wörter geben).
- Das **Vokabular** ist die Menge aller Zeichen, die im Datensatz vorkommen.

In [ ]:
# Alle Stücke zu einem langen Text zusammenfügen.
# Eine Leerzeile trennt die Stücke – so lernt das Modell auch, wo ein Stück endet.
text = "\n\n".join(stuecke)
print(f"Gesamtlänge des Textes: {len(text):,} Zeichen")

# Vokabular = alle vorkommenden Zeichen (sortiert, damit reproduzierbar)
vocab = sorted(set(text))
vocab_groesse = len(vocab)
print(f"Vokabulargröße: {vocab_groesse} Zeichen")

# Nachschlagetabellen: Zeichen -> ID und ID -> Zeichen
char2idx = {zeichen: idx for idx, zeichen in enumerate(vocab)}
idx2char = {idx: zeichen for idx, zeichen in enumerate(vocab)}

def encode(s):
    """Text -> Liste von Zahlen-IDs"""
    return [char2idx[zeichen] for zeichen in s]

def decode(ids):
    """Liste von Zahlen-IDs -> Text"""
    return "".join(idx2char[i] for i in ids)

# Kleiner Test: Encoding und Decoding müssen sich gegenseitig aufheben
beispiel = "K:D |: ABC"
print("Encoding :", encode(beispiel))
print("Decoding :", decode(encode(beispiel)))

### Sequenzen mit gleitendem Fenster

Das Modell soll das **nächste Zeichen** vorhersagen. Dafür schieben wir ein Fenster der Länge `SEQ_LEN` über den Text:

- **Input:** Zeichen `i` bis `i + SEQ_LEN`
- **Ziel:** dieselben Zeichen, **um 1 verschoben** (also an jeder Position das jeweils nächste Zeichen)

Beispiel mit `SEQ_LEN = 4`:  Input `K:D␣` → Ziel `:D␣|`

Damit nicht zu viele fast identische Sequenzen entstehen, verschieben wir das Fenster jeweils um `SCHRITT` Zeichen. Danach teilen wir die Sequenzen in **Train (80 %) / Validation (10 %) / Test (10 %)** auf.

In [ ]:
# Kompletten Text einmal encodieren
daten_ids = torch.tensor(encode(text), dtype=torch.long)

# Gleitendes Fenster: Input = Fenster, Ziel = Fenster um 1 Zeichen verschoben
eingaben, ziele = [], []
for start in range(0, len(daten_ids) - SEQ_LEN - 1, SCHRITT):
    eingaben.append(daten_ids[start : start + SEQ_LEN])
    ziele.append(daten_ids[start + 1 : start + SEQ_LEN + 1])

X = torch.stack(eingaben)   # Form: (Anzahl Sequenzen, SEQ_LEN)
Y = torch.stack(ziele)
print("Alle Sequenzen:", tuple(X.shape))

# Zufällig mischen und in Train/Val/Test aufteilen (80/10/10)
indizes = torch.randperm(len(X))
X, Y = X[indizes], Y[indizes]

n_train = int(0.8 * len(X))
n_val = int(0.1 * len(X))

train_daten = TensorDataset(X[:n_train], Y[:n_train])
val_daten = TensorDataset(X[n_train : n_train + n_val], Y[n_train : n_train + n_val])
test_daten = TensorDataset(X[n_train + n_val :], Y[n_train + n_val :])

train_loader = DataLoader(train_daten, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_daten, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_daten, batch_size=BATCH_SIZE)

print(f"Train: {len(train_daten):,} | Val: {len(val_daten):,} | Test: {len(test_daten):,} Sequenzen")

## 4. Task 2 – Das Modell


1. **`nn.Embedding`** – wandelt jede Zeichen-ID in einen lernbaren Vektor um
2. **`nn.LSTM`** – verarbeitet die Sequenz Schritt für Schritt und merkt sich den Kontext im Hidden State (die Gates des LSTM lösen das Vanishing-Gradient-Problem des einfachen RNN)
3. **`nn.Linear`** – berechnet für jeden Zeitschritt einen Score (Logit) pro Zeichen im Vokabular

Über den Parameter `rnn_typ` können wir dieselbe Klasse auch mit `GRU` oder einfachem `RNN` benutzen → das brauchen wir für die Ablationsstudie (Bonus 1).

In [ ]:
class MusikRNN(nn.Module):
    """Sprachmodell für ABC-Musik: Embedding -> RNN/LSTM/GRU -> Linear"""

    def __init__(self, vocab_groesse, embed_dim, hidden_size, num_layers, rnn_typ="LSTM"):
        super().__init__()
        self.embedding = nn.Embedding(vocab_groesse, embed_dim)

        # Der gewünschte rekurrente Layer (alle mit batch_first=True wie in der Vorlesung)
        if rnn_typ == "LSTM":
            self.rnn = nn.LSTM(embed_dim, hidden_size, num_layers, batch_first=True)
        elif rnn_typ == "GRU":
            self.rnn = nn.GRU(embed_dim, hidden_size, num_layers, batch_first=True)
        else:
            self.rnn = nn.RNN(embed_dim, hidden_size, num_layers, batch_first=True)

        self.fc = nn.Linear(hidden_size, vocab_groesse)

    def forward(self, x, hidden=None):
        # x: (Batch, SeqLen) mit Zeichen-IDs
        emb = self.embedding(x)              # -> (Batch, SeqLen, EmbedDim)
        out, hidden = self.rnn(emb, hidden)  # -> (Batch, SeqLen, HiddenSize)
        logits = self.fc(out)                # -> (Batch, SeqLen, VocabGroesse)
        return logits, hidden


# Kurzer Blick auf die Architektur
test_modell = MusikRNN(vocab_groesse, EMBED_DIM, HIDDEN_SIZE, NUM_LAYERS)
print(test_modell)
anzahl_parameter = sum(p.numel() for p in test_modell.parameters())
print(f"Anzahl Parameter: {anzahl_parameter:,}")

## 5. Task 3 – Training

- **Loss:** `CrossEntropyLoss` über alle Zeitschritte (an jeder Position soll das nächste Zeichen vorhergesagt werden)
- **Optimizer:** `Adam`
- **Metriken (Definition vom Übungsblatt):** pro Sequenz betrachten wir die Vorhersage für das **nächste Zeichen nach der Sequenz** (letzter Zeitschritt):
  - **Top-1 Accuracy** = Anteil der Sequenzen, bei denen das richtige Zeichen die wahrscheinlichste Vorhersage ist
  - **Top-5 Accuracy** = Anteil der Sequenzen, bei denen das richtige Zeichen unter den 5 wahrscheinlichsten ist
- Nach jeder Epoche loggen wir Loss und beide Accuracies (Train + Validation) zu **wandb**.

In [ ]:
def run_epoch(modell, loader, criterion, optimizer=None):
    """Eine Epoche: mit optimizer -> Training, ohne optimizer -> nur Auswertung.
    Gibt (loss, top1_accuracy, top5_accuracy) zurück."""
    training = optimizer is not None
    modell.train() if training else modell.eval()

    gesamt_loss = 0.0
    korrekt_top1 = 0
    korrekt_top5 = 0
    anzahl = 0

    # Im Evaluationsmodus brauchen wir keine Gradienten
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            logits, _ = modell(x)  # (Batch, SeqLen, Vocab)

            # CrossEntropy erwartet 2D-Logits: alle Zeitschritte "flach" machen
            loss = criterion(logits.reshape(-1, vocab_groesse), y.reshape(-1))

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            # --- Metriken: Vorhersage des nächsten Zeichens (letzter Zeitschritt) ---
            letzte_logits = logits[:, -1, :]             # (Batch, Vocab)
            ziel = y[:, -1]                              # das tatsächliche nächste Zeichen
            top5 = letzte_logits.topk(5, dim=1).indices  # die 5 wahrscheinlichsten IDs

            korrekt_top1 += (top5[:, 0] == ziel).sum().item()
            korrekt_top5 += (top5 == ziel.unsqueeze(1)).any(dim=1).sum().item()

            gesamt_loss += loss.item() * x.size(0)
            anzahl += x.size(0)

    return gesamt_loss / anzahl, korrekt_top1 / anzahl, korrekt_top5 / anzahl

In [ ]:
def trainiere_modell(rnn_typ, run_id, changed_params):
    """Trainiert ein Modell und loggt alles zu wandb. Gibt (modell, historie) zurück."""

    # Run-Name nach dem Schema aus dem wandb-Guide
    run_name = f"run{run_id}_{changed_params}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    modell = MusikRNN(vocab_groesse, EMBED_DIM, HIDDEN_SIZE, NUM_LAYERS, rnn_typ).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(modell.parameters(), lr=LEARNING_RATE)

    run_config = dict(config)
    run_config["model"] = rnn_typ
    run_config["run_id"] = run_id

    historie = {"train_loss": [], "val_loss": [],
                "train_top1": [], "val_top1": [],
                "train_top5": [], "val_top5": []}

    # Hinweis: das früher unter Windows nötige start_method="thread" ist in
    # aktuellen wandb-Versionen deprecated und nicht mehr erforderlich.
    with wandb.init(entity=ENTITY, project=PROJECT, name=run_name, config=run_config) as run:

        for epoche in range(1, NUM_EPOCHS + 1):
            train_loss, train_top1, train_top5 = run_epoch(modell, train_loader, criterion, optimizer)
            val_loss, val_top1, val_top5 = run_epoch(modell, val_loader, criterion)

            historie["train_loss"].append(train_loss)
            historie["val_loss"].append(val_loss)
            historie["train_top1"].append(train_top1)
            historie["val_top1"].append(val_top1)
            historie["train_top5"].append(train_top5)
            historie["val_top5"].append(val_top5)

            # DAS erscheint als Kurven in wandb
            run.log({
                "epoch": epoche,
                "train/loss": train_loss,
                "train/top1_accuracy": train_top1,
                "train/top5_accuracy": train_top5,
                "val/loss": val_loss,
                "val/top1_accuracy": val_top1,
                "val/top5_accuracy": val_top5,
                "lr": optimizer.param_groups[0]["lr"],
            })

            print(f"[{rnn_typ}] Epoche {epoche:2d}/{NUM_EPOCHS} | "
                  f"Train-Loss {train_loss:.4f} | Val-Loss {val_loss:.4f} | "
                  f"Val Top-1 {val_top1:.2%} | Val Top-5 {val_top5:.2%}")

        # --- Task 5: finale Evaluation auf dem Test-Set (noch im selben wandb-Run) ---
        test_loss, test_top1, test_top5 = run_epoch(modell, test_loader, criterion)
        run.log({"test/loss": test_loss,
                 "test/top1_accuracy": test_top1,
                 "test/top5_accuracy": test_top5,
                 "best_val_loss": min(historie["val_loss"])})

        historie["test"] = {"loss": test_loss, "top1": test_top1, "top5": test_top5}
        print(f"\n[{rnn_typ}] TEST: Loss {test_loss:.4f} | Top-1 {test_top1:.2%} | Top-5 {test_top5:.2%}")

    return modell, historie

In [ ]:
# Haupt-Training: LSTM (Run 1)
modell_lstm, historie_lstm = trainiere_modell(
    "LSTM", run_id=1, changed_params=f"lstm_hidden{HIDDEN_SIZE}_lr{LEARNING_RATE}")

# Gewichte speichern (gehören mit in die Abgabe)
torch.save(modell_lstm.state_dict(), "musik_rnn_lstm.pt")
print("Gewichte gespeichert: musik_rnn_lstm.pt")

[LSTM] Epoche  1/10 | Train-Loss 2.1715 | Val-Loss 1.5480 | Val Top-1 54.26% | Val Top-5 84.90%


[LSTM] Epoche  2/10 | Train-Loss 1.4238 | Val-Loss 1.3472 | Val Top-1 59.77% | Val Top-5 88.17%


[LSTM] Epoche  3/10 | Train-Loss 1.2945 | Val-Loss 1.2597 | Val Top-1 62.00% | Val Top-5 90.23%


[LSTM] Epoche  4/10 | Train-Loss 1.2245 | Val-Loss 1.2049 | Val Top-1 63.52% | Val Top-5 90.96%


[LSTM] Epoche  5/10 | Train-Loss 1.1785 | Val-Loss 1.1712 | Val Top-1 64.57% | Val Top-5 91.84%


...

wandb: uploading data


wandb: 
wandb: Run history:
wandb:       best_val_loss ▁
wandb:               epoch ▁▂▃▃▄▅▆▆▇█
wandb:                  lr ▁▁▁▁▁▁▁▁▁▁
wandb:           test/loss ▁
wandb:  test/top1_accuracy ▁
wandb:  test/top5_accuracy ▁
wandb:          train/loss █▃▂▂▂▂▁▁▁▁
wandb: train/top1_accuracy ▁▅▆▆▇▇▇███
wandb: train/top5_accuracy ▁▆▇▇▇█████
wandb:            val/loss █▅▄▃▂▂▂▁▁▁
wandb:                  +2 ...
wandb: 
wandb: Run summary:
wandb:       best_val_loss 1.08239
wandb:               epoch 10
wandb:                  lr 0.001
wandb:           test/loss 1.08056
wandb:  test/top1_accuracy 0.6758
wandb:  test/top5_accuracy 0.93362
wandb:          train/loss 1.0635
wandb: train/top1_accuracy 0.68397
wandb: train/top5_accuracy 0.93729
wandb:            val/loss 1.08239
wandb:                  +2 ...
wandb: 


wandb:  View run run1_lstm_hidden256_lr0.001_20260716_172736 at: https://wandb.ai/eidl-thm/4.block_Jordan_Pokem_Leslie_Tsafack_RNN/runs/sjxjbcmh
wandb:  View project at: https://wandb.ai/eidl-thm/4.block_Jordan_Pokem_Leslie_Tsafack_RNN
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: .\wandb\run-20260716_172738-sjxjbcmh\logs


Gewichte gespeichert: musik_rnn_lstm.pt


## 6. Task 5 – Evaluation auf dem Test-Set

Die finale Auswertung auf dem **Test-Set** (Daten, die das Modell nie gesehen hat) wurde direkt am Ende des Trainings durchgeführt, damit sie im selben wandb-Run landet. Hier noch einmal übersichtlich:

In [ ]:
print("Ergebnisse LSTM auf dem Test-Set")
print(f"  Top-1 Accuracy: {historie_lstm['test']['top1']:.2%}")
print(f"  Top-5 Accuracy: {historie_lstm['test']['top5']:.2%}")
print(f"  Loss:           {historie_lstm['test']['loss']:.4f}")
print(f"\nZum Vergleich: zufälliges Raten hätte Top-1 ≈ {1 / vocab_groesse:.2%}")

## 7. Task 4 – Musikgenerierung

Jetzt lassen wir das Modell selbst Musik schreiben – **Zeichen für Zeichen**:

1. Wir geben eine **Seed-Sequenz** vor (ein typischer ABC-Anfang mit Taktart und Tonart)
2. Das Modell berechnet für das nächste Zeichen eine Wahrscheinlichkeitsverteilung (**Softmax** über die Logits)
3. Wir **ziehen eine Stichprobe** aus dieser Verteilung (`torch.multinomial`) – so klingt jedes generierte Stück anders. (Immer nur das wahrscheinlichste Zeichen zu nehmen würde sehr repetitive Musik ergeben.)
4. Das gezogene Zeichen wird angehängt und wieder ins Modell gesteckt (der Kontext steckt im Hidden State) – so lange, bis das Stück fertig ist

In [ ]:
def generiere_musik(modell, seed, laenge=GEN_LAENGE):
    """Generiert Musik in ABC-Notation, beginnend mit der Seed-Sequenz."""
    modell.eval()

    ids = encode(seed)
    ergebnis = list(ids)

    eingabe = torch.tensor([ids], device=DEVICE)  # Form: (1, SeedLaenge)
    hidden = None

    with torch.no_grad():
        # 1) Seed durch das Modell schicken, um den Hidden State aufzubauen
        logits, hidden = modell(eingabe, hidden)

        # 2) Zeichen für Zeichen weiterschreiben
        for _ in range(laenge):
            wahrscheinlichkeiten = torch.softmax(logits[:, -1, :], dim=1)
            naechste_id = torch.multinomial(wahrscheinlichkeiten, num_samples=1)
            ergebnis.append(naechste_id.item())
            # nur das neue Zeichen einspeisen – der Kontext steckt im Hidden State
            logits, hidden = modell(naechste_id, hidden)

    return decode(ergebnis)


def schneide_stueck(abc_text):
    """Schneidet am Ende des ersten Stücks ab (Leerzeile = neues Stück beginnt)."""
    pos = abc_text.find("\n\n", 20)
    return abc_text[:pos] if pos != -1 else abc_text

In [ ]:
# Typischer Anfang eines irischen Stücks als Seed (Kopfzeilen wie im Datensatz)
SEED = "X:1\nL:1/8\nM:6/8\nK:D\n"

generierte_stuecke = []
for i in range(3):
    stueck = schneide_stueck(generiere_musik(modell_lstm, SEED))
    generierte_stuecke.append(stueck)
    print(f"=== Generiertes Stück {i + 1} ===")
    print(stueck)
    print()

=== Generiertes Stück 2 ===
X:1
L:1/8
M:6/8
K:D
 g2 a fed | Aaa a^ga | fed f2 e | fdf edB | dAG FED | AFD DFA |1 fAB d3 | f2 d e2 g | fed fed | fdB gec | dBG F2 A :|2 def gdB | 
 ABA FDF GEA || dBB fBB | fBd Bdg | fdB BAF | EFE EFA | FGE EFG ||



=== Generiertes Stück 3 ===
X:1
L:1/8
M:6/8
K:D
 F | ADF FDF | G D2 EDC | DFE{F} ABA | dcA BGE | FDF Adf | e2 c CFB, | CEE Acd | edc BED | D2 ^G A3 |]



## 8. Bonus 1 – Ablationsstudie: LSTM vs. GRU (2 Punkte)

Die **GRU** (Gated Recurrent Unit) ist die "kleine Schwester" des LSTM: ebenfalls Gates gegen das Vanishing-Gradient-Problem, aber mit weniger Parametern (kein separater Cell State).

Wir trainieren **exakt dasselbe Modell** noch einmal – nur der rekurrente Layer wird getauscht (`rnn_typ="GRU"`). Beide Runs landen im selben wandb-Projekt und lassen sich dort direkt vergleichen.

In [ ]:
# Zweiter Run: gleiche Hyperparameter, nur GRU statt LSTM
modell_gru, historie_gru = trainiere_modell(
    "GRU", run_id=2, changed_params=f"gru_hidden{HIDDEN_SIZE}_lr{LEARNING_RATE}")

torch.save(modell_gru.state_dict(), "musik_rnn_gru.pt")
print("Gewichte gespeichert: musik_rnn_gru.pt")

[GRU] Epoche  1/10 | Train-Loss 1.9190 | Val-Loss 1.4225 | Val Top-1 57.84% | Val Top-5 87.08%


[GRU] Epoche  2/10 | Train-Loss 1.3175 | Val-Loss 1.2520 | Val Top-1 62.36% | Val Top-5 89.97%


[GRU] Epoche  3/10 | Train-Loss 1.2115 | Val-Loss 1.1864 | Val Top-1 63.54% | Val Top-5 91.47%


[GRU] Epoche  4/10 | Train-Loss 1.1577 | Val-Loss 1.1499 | Val Top-1 65.19% | Val Top-5 91.88%


[GRU] Epoche  5/10 | Train-Loss 1.1221 | Val-Loss 1.1224 | Val Top-1 66.54% | Val Top-5 92.61%


...

wandb: uploading history steps 8-10, summary


wandb: 
wandb: Run history:
wandb:       best_val_loss ▁
wandb:               epoch ▁▂▃▃▄▅▆▆▇█
wandb:                  lr ▁▁▁▁▁▁▁▁▁▁
wandb:           test/loss ▁
wandb:  test/top1_accuracy ▁
wandb:  test/top5_accuracy ▁
wandb:          train/loss █▃▂▂▂▂▁▁▁▁
wandb: train/top1_accuracy ▁▅▆▆▇▇▇███
wandb: train/top5_accuracy ▁▆▇▇▇█████
wandb:            val/loss █▅▃▃▂▂▂▁▁▁
wandb:                  +2 ...
wandb: 
wandb: Run summary:
wandb:       best_val_loss 1.06
wandb:               epoch 10
wandb:                  lr 0.001
wandb:           test/loss 1.05845
wandb:  test/top1_accuracy 0.68544
wandb:  test/top5_accuracy 0.9364
wandb:          train/loss 1.02334
wandb: train/top1_accuracy 0.69715
wandb: train/top5_accuracy 0.94147
wandb:            val/loss 1.06
wandb:                  +2 ...
wandb: 


wandb:  View run run2_gru_hidden256_lr0.001_20260716_172935 at: https://wandb.ai/eidl-thm/4.block_Jordan_Pokem_Leslie_Tsafack_RNN/runs/v105fm1d
wandb:  View project at: https://wandb.ai/eidl-thm/4.block_Jordan_Pokem_Leslie_Tsafack_RNN
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: .\wandb\run-20260716_172935-v105fm1d\logs


Gewichte gespeichert: musik_rnn_gru.pt


In [ ]:
# Direkter Vergleich auf dem Test-Set
print(f"{'Modell':8} | {'Test-Loss':>9} | {'Top-1':>7} | {'Top-5':>7}")
print("-" * 42)
for name, hist in [("LSTM", historie_lstm), ("GRU", historie_gru)]:
    t = hist["test"]
    print(f"{name:8} | {t['loss']:9.4f} | {t['top1']:7.2%} | {t['top5']:7.2%}")

## 9. Bonus 2 – Validity Check: die "Grammatik" der ABC-Notation (2 Punkte)

Top-1 Accuracy sagt nichts darüber aus, ob ein **ganzes Stück** gültige ABC-Notation ist. Deshalb prüfen wir die generierten Stücke mit einem kleinen Regel-Algorithmus:

| Regel | Warum? |
|---|---|
| Kopfzeilen vorhanden (z. B. `M:`, `L:`, `K:`) | Jedes ABC-Stück braucht einen Header |
| Tonart `K:` vorhanden | Ohne Tonart ist das Stück nicht spielbar |
| Genügend Taktstriche (`\|`) | Musik ist in Takte gegliedert |
| Enthält Noten (A–G / a–g) | Sonst ist es keine Melodie |
| Takte sind nicht leer | Leere Takte deuten auf kaputte Syntax hin |
| Endet mit Schlussstrich (`\|`, `\|]` oder `:\|`) | Das Stück soll "logisch" enden |

In [ ]:
def pruefe_abc_syntax(stueck):
    """Prüft einfache Grammatik-Regeln der ABC-Notation.
    Gibt ein Dictionary {Regel: bestanden (True/False)} zurück."""
    zeilen = stueck.strip().split("\n")

    # Kopfzeilen erkennen wir an "Buchstabe:" am Zeilenanfang (z. B. "K:D")
    kopfzeilen = [z for z in zeilen if len(z) >= 2 and z[1] == ":"]
    melodie = "".join(z for z in zeilen if z not in kopfzeilen)
    takte = [t for t in melodie.split("|") if t.strip()]

    return {
        "Kopfzeilen vorhanden": len(kopfzeilen) >= 2,
        "Tonart (K:) vorhanden": any(z.startswith("K:") for z in zeilen),
        "genug Taktstriche (>= 4)": melodie.count("|") >= 4,
        "enthaelt Noten (A-G/a-g)": any(c in "ABCDEFGabcdefg" for c in melodie),
        "Takte nicht leer": len(takte) >= 4,
        "endet mit Schlussstrich": melodie.rstrip().endswith(("|", "|]", ":|")),
    }


# Alle generierten Stücke prüfen
alle_checks = []
for i, stueck in enumerate(generierte_stuecke, start=1):
    check = pruefe_abc_syntax(stueck)
    alle_checks.append(check)
    print(f"Stück {i}: {sum(check.values())}/{len(check)} Regeln erfüllt")
    for regel, ok in check.items():
        print(f"   {'[OK]  ' if ok else '[FEHLT]'} {regel}")
    print()

# Gesamt-Score: Anteil erfüllter Regeln über alle Stücke
validity_score = sum(sum(c.values()) for c in alle_checks) / (len(alle_checks) * len(alle_checks[0]))
print(f"Validity-Score gesamt: {validity_score:.0%}")

## 10. Bonus 3 – Kreativität: die generierte Musik hörbar machen (1 Punkt)

Wir übersetzen ein generiertes Stück in eine **WAV-Audiodatei** – ganz ohne Zusatz-Bibliotheken:

1. Ein Mini-Parser liest die Noten aus der ABC-Notation (Notenname, Oktave, Länge, Pausen)
2. Jede Note wird als **Sinuswelle** mit der passenden Frequenz erzeugt (Kammerton A4 = 440 Hz)
3. Alles wird aneinandergehängt und als `.wav` gespeichert → direkt im Notebook abspielbar

*(Der Parser ist bewusst einfach gehalten und ignoriert z. B. Verzierungen und Akkorde – für einen Hör-Eindruck reicht es!)*

In [ ]:
from IPython.display import Audio

# Halbtonabstände der Stammtöne zur Note C
NOTEN_HALBTOENE = {"C": 0, "D": 2, "E": 4, "F": 5, "G": 7, "A": 9, "B": 11}


def abc_zu_toenen(stueck):
    """Sehr einfacher ABC-Parser: liefert eine Liste (Frequenz in Hz, Dauer in Einheiten).
    Frequenz 0 bedeutet Pause."""
    # Kopfzeilen (K:, M:, ...) enthalten keine Melodie -> überspringen
    zeilen = [z for z in stueck.split("\n") if not (len(z) >= 2 and z[1] == ":")]
    melodie = " ".join(zeilen)

    toene = []
    i = 0
    while i < len(melodie):
        zeichen = melodie[i]

        # Vorzeichen: ^ = Halbton hoch, _ = Halbton runter, = = Auflösung
        versatz = 0
        if zeichen in "^_=":
            versatz = {"^": 1, "_": -1, "=": 0}[zeichen]
            i += 1
            if i >= len(melodie):
                break
            zeichen = melodie[i]

        if zeichen.upper() in NOTEN_HALBTOENE:
            halbton = NOTEN_HALBTOENE[zeichen.upper()] + versatz
            oktave = 5 if zeichen.islower() else 4  # Kleinbuchstaben = eine Oktave höher
            i += 1
            # Oktav-Zeichen: ' = höher, , = tiefer
            while i < len(melodie) and melodie[i] in "',":
                oktave += 1 if melodie[i] == "'" else -1
                i += 1
            # Notenlänge: Zahl = Vielfaches, /2 = halbe Länge usw.
            dauer = 1.0
            if i < len(melodie) and melodie[i].isdigit():
                dauer = float(melodie[i])
                i += 1
            if i < len(melodie) and melodie[i] == "/":
                i += 1
                teiler = 2.0
                if i < len(melodie) and melodie[i].isdigit():
                    teiler = float(melodie[i])
                    i += 1
                dauer /= teiler
            # Frequenz über die Standard-Stimmung (A4 = 440 Hz)
            midi_nummer = 12 * (oktave + 1) + halbton
            frequenz = 440.0 * 2 ** ((midi_nummer - 69) / 12)
            toene.append((frequenz, dauer))
        elif zeichen in "zZx":  # Pausen
            i += 1
            dauer = 1.0
            if i < len(melodie) and melodie[i].isdigit():
                dauer = float(melodie[i])
                i += 1
            toene.append((0.0, dauer))
        else:
            i += 1  # alles andere (Taktstriche, Leerzeichen, ...) überspringen
    return toene


def toene_zu_wav(toene, dateiname, einheit_sekunden=0.2, abtastrate=22050):
    """Erzeugt aus (Frequenz, Dauer)-Paaren eine WAV-Datei mit Sinustönen."""
    signal = []
    for frequenz, dauer in toene:
        n = int(dauer * einheit_sekunden * abtastrate)
        if n <= 0:
            continue
        t = np.arange(n) / abtastrate
        welle = 0.4 * np.sin(2 * np.pi * frequenz * t) if frequenz > 0 else np.zeros(n)
        # kurzes Ein-/Ausblenden gegen Knackser
        fade = min(200, n // 2)
        if fade > 0:
            welle[:fade] = welle[:fade] * np.linspace(0, 1, fade)
            welle[-fade:] = welle[-fade:] * np.linspace(1, 0, fade)
        signal.append(welle)

    signal = np.concatenate(signal)
    # in 16-Bit-Ganzzahlen umwandeln und mit dem Standard-Modul "wave" speichern
    signal_int16 = (signal * 32767).astype(np.int16)
    with wave.open(dateiname, "wb") as w:
        w.setnchannels(1)   # Mono
        w.setsampwidth(2)   # 16 Bit = 2 Byte
        w.setframerate(abtastrate)
        w.writeframes(signal_int16.tobytes())
    return dateiname


toene = abc_zu_toenen(generierte_stuecke[0])
print(f"{len(toene)} Töne/Pausen aus dem Stück gelesen")

if toene:
    wav_datei = toene_zu_wav(toene, "generierte_musik.wav")
    print(f"Gespeichert: {wav_datei}")
else:
    print("Keine Noten erkannt – bitte ein anderes Stück versuchen.")

Audio(wav_datei)  # Abspielen direkt im Notebook

## 11. Trainingskurven & Export für die Präsentation

Die interaktiven Kurven liegen in wandb – hier erzeugen wir zusätzlich statische Plots (für den One-Pager und die PowerPoint-Präsentation) und speichern alle Ergebnisse als Dateien.

In [ ]:
# Farben und einheitlicher, dezenter Stil für alle Plots
BLAU = "#2a78d6"
GRUEN = "#008300"

epochen = list(range(1, NUM_EPOCHS + 1))

def stil(ax, titel, y_beschriftung, x_beschriftung="Epoche"):
    ax.set_title(titel, fontsize=12, color="#0b0b0b")
    ax.set_xlabel(x_beschriftung, color="#52514e")
    ax.set_ylabel(y_beschriftung, color="#52514e")
    ax.grid(color="#e1e0d9", linewidth=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)

# --- Plot 1: Loss-Kurven (LSTM) ---
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochen, historie_lstm["train_loss"], color=BLAU, linewidth=2, label="Training")
ax.plot(epochen, historie_lstm["val_loss"], color=GRUEN, linewidth=2, label="Validierung")
ax.set_xticks(epochen)
stil(ax, "LSTM – Loss pro Epoche", "CrossEntropy-Loss")
fig.tight_layout()
fig.savefig("kurven_loss.png", dpi=150)
plt.show()

# --- Plot 2: Accuracy-Kurven (LSTM, Validierung) ---
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochen, historie_lstm["val_top1"], color=BLAU, linewidth=2, label="Top-1 (Validierung)")
ax.plot(epochen, historie_lstm["val_top5"], color=GRUEN, linewidth=2, label="Top-5 (Validierung)")
ax.set_xticks(epochen)
stil(ax, "LSTM – Accuracy pro Epoche", "Accuracy")
fig.tight_layout()
fig.savefig("kurven_accuracy.png", dpi=150)
plt.show()

# --- Plot 3: Vergleich LSTM vs. GRU auf dem Test-Set ---
fig, ax = plt.subplots(figsize=(7, 4))
metriken = ["Top-1", "Top-5"]
lstm_werte = [historie_lstm["test"]["top1"], historie_lstm["test"]["top5"]]
gru_werte = [historie_gru["test"]["top1"], historie_gru["test"]["top5"]]
x_pos = np.arange(len(metriken))
breite = 0.33  # etwas schmaler als der Abstand -> kleine Lücke zwischen den Balken
balken1 = ax.bar(x_pos - 0.185, lstm_werte, breite, label="LSTM", color=BLAU)
balken2 = ax.bar(x_pos + 0.185, gru_werte, breite, label="GRU", color=GRUEN)
ax.bar_label(balken1, labels=[f"{w:.1%}" for w in lstm_werte], color="#52514e")
ax.bar_label(balken2, labels=[f"{w:.1%}" for w in gru_werte], color="#52514e")
ax.set_xticks(x_pos, metriken)
stil(ax, "LSTM vs. GRU – Accuracy auf dem Test-Set", "Accuracy", x_beschriftung="Metrik")
fig.tight_layout()
fig.savefig("vergleich_lstm_gru.png", dpi=150)
plt.show()

In [ ]:
# Alle Ergebnisse als JSON – daraus baut erstelle_praesentation.py die PowerPoint
ergebnisse = {
    "config": config,
    "vocab_groesse": vocab_groesse,
    "anzahl_sequenzen": int(len(X)),
    "anzahl_parameter": anzahl_parameter,
    "lstm": historie_lstm,
    "gru": historie_gru,
    "validity_score": validity_score,
    "beispiel_original": stuecke[0],
    "seed": SEED,
}
with open("ergebnisse.json", "w", encoding="utf-8") as f:
    json.dump(ergebnisse, f, ensure_ascii=False, indent=2)

# Generierte Stücke als Textdatei (für Abgabe und One-Pager)
with open("beispiele.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(generierte_stuecke))

print("Gespeichert: ergebnisse.json, beispiele.txt")
print("PowerPoint erzeugen mit:  python erstelle_praesentation.py")

## 12. Fazit

**Was wir gemacht haben:**
- ABC-Notation als Text betrachtet, mit einem **Zeichen-Tokenizer** encodiert und per gleitendem Fenster Trainingssequenzen erstellt (Task 1)
- Ein Sprachmodell **Embedding → LSTM → Linear** implementiert (Task 2) und mit CrossEntropy + Adam trainiert, komplett mit **wandb** getrackt (Task 3)
- Auf dem Test-Set mit **Top-1 / Top-5 Accuracy** evaluiert (Task 5)
- Neue Melodien durch **autoregressives Sampling** (Softmax + Stichprobe) generiert (Task 4)
- **Bonus:** GRU-Ablation, ABC-Grammatik-Check und eine hörbare WAV-Ausgabe

**Beobachtungen / Herausforderungen:**
- Die größte Stellschraube war **Datenmenge vs. Trainingszeit** – über `NUM_TUNES` und `SCHRITT` lässt sich der Kompromiss steuern.
- Die generierten Stücke sind syntaktisch meist korrekt (siehe Validity-Score); sehr lange Stücke verlieren manchmal die Struktur, weil das Modell beim Training nur `SEQ_LEN` Zeichen Kontext gesehen hat.
- **Sampling statt Argmax** war wichtig: mit Argmax wiederholt sich die Musik sehr schnell.
